# Technical Notes

LLMs process text by splitting it into smaller units called tokens rather than working directly with words.

- **Token:** A word, part of a word, or punctuation mark. Token ≠ Word.
- **Tokenizer:** Converts text into token IDs.
- **Tokenization:** The process of splitting text into tokens.
- **Embedding:** Numerical vector representations of token IDs.
- **Context Window:** The maximum number of tokens a model can process within a context.
- **Input Tokens:** Tokens sent to the model.
- **Output Tokens:** Tokens generated by the model.
- **Special Tokens:** Define message boundaries, roles, and control flow.

## LLM Pipeline

`Text → Tokenizer → Token IDs → Embeddings → Transformer → Output`

## Engineering Perspective

- Token count directly affects API costs.
- Context window management is important for RAG and long-prompt applications.
- Unnecessary tokens can increase cost and processing time.
- Different models may use different tokenizers.
- Input and output token usage should be tracked separately.

## Tools
- Tiktoken
- Hugging Face Tokenizers

In [16]:
from pydantic import BaseModel, Field
from typing import List
class AIResponse(BaseModel):

    answer: str = Field(description="The answer generated by the AI model.", min_length=1, max_length=1000)
    confidence: float = Field(description="The confidence level of the AI model's answer.", gt= 0.0, lt=1.0)
    sources: List[str] = Field(description="The sources used to generate the answer.")
    question : str | None = None

ai_response = AIResponse(
    answer = "Ankara Türkiye'nin başkentidir.",
    confidence = 0.95,
    sources = ["https://tr.wikipedia.org/wiki/Ankara"])

print(ai_response.answer)
print(ai_response.model_dump())

Ankara Türkiye'nin başkentidir.
{'answer': "Ankara Türkiye'nin başkentidir.", 'confidence': 0.95, 'sources': ['https://tr.wikipedia.org/wiki/Ankara'], 'question': None}


In [13]:
data = {
    "answer": "Ankara Türkiye'nin başkentidir.",
    "confidence": 0.95,
    "sources": ["https://tr.wikipedia.org/wiki/Ankara"]
}
ai_response = AIResponse.model_validate(data)
ai_response

AIResponse(answer="Ankara Türkiye'nin başkentidir.", confidence=0.95, sources=['https://tr.wikipedia.org/wiki/Ankara'])

In [14]:
data = {
    "answer": "Ankara Türkiye'nin başkentidir.",
    "confidence": 1.95,
    "sources": ["https://tr.wikipedia.org/wiki/Ankara"]
}
ai_response = AIResponse.model_validate(data)
ai_response

ValidationError: 1 validation error for AIResponse
confidence
  Input should be less than 1 [type=less_than, input_value=1.95, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than

In [33]:
from pydantic import BaseModel, field_validator

class Book(BaseModel):
    title: str
    author: str | None = None
    page_count: int | None = None
    library: str | None = "Gece Yarısı Kütüphanesi"


    @field_validator("page_count")
    @classmethod
    def check_page_count(cls, value):
        if value is not None and value <= 0:
            raise ValueError("Page count must be a positive integer.")
        return value

    @classmethod
    def get_library(cls):
        return cls.library
    
    def get_title(self):
        return self.title

book = Book(title="The Great Gatsby")

In [21]:
book = Book(title="The Great Gatsby", page_count=0)

ValidationError: 1 validation error for Book
page_count
  Value error, Page count must be a positive integer. [type=value_error, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error

In [30]:
book.get_title()

'The Great Gatsby'

In [35]:
Book.get_library()

AttributeError: library

In [40]:
class Library:
    name = "Gece Yarısı Kütüphanesi"

    def __init__(self,name):
        self.title= title

    @classmethod
    def get_library_name(cls):
        return cls.name

    #instance method
    def get_title(self):
        return self.title

Library.get_library_name()

'Gece Yarısı Kütüphanesi'

# Pydantic – Quick Notes

Pydantic is a Python library used for data validation, type checking, and structured data management.

## Main Purpose
- Validate incoming data.
- Enforce data types and constraints.
- Define structured data schemas.
- Convert and serialize Python objects.
- Catch invalid or missing data.

## Core Features

| Feature | Purpose |
|---|---|
| `BaseModel` | Define data schemas |
| `Field` | Add constraints and descriptions |
| `model_validate()` | Validate input data |
| `model_dump()` | Convert model to dictionary |
| `model_dump_json()` | Serialize model to JSON |
| `Optional` | Allow nullable fields |
| `@field_validator` | Define custom validation rules |
| `@classmethod` | Define class-level methods |
| `self` | Access instance attributes |
| `cls` | Access class attributes |

## When to Use Pydantic?
- **FastAPI:** Request and response validation.
- **LLM Applications:** Validate structured LLM outputs.
- **RAG:** Validate document metadata.
- **AI Agents:** Define tool input/output schemas.
- **Data Pipelines:** Ensure data consistency.
- **Configuration:** Validate application settings.

**Key Takeaway:** Pydantic helps build reliable LLM applications by ensuring that data follows predefined schemas and validation rules. Pydantic is a data validation layer that improves reliability, consistency, and type safety in Python applications.



# FastAPI & Uvicorn

FastAPI is a Python framework for building REST APIs, while Uvicorn is an ASGI server that runs FastAPI applications and handles HTTP requests.

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    return {"message": "Hello World!"}
```

**Installation:**
```bash
pip install fastapi uvicorn
```

**Run the application:**
```bash
uvicorn main:app --reload
```

- `FastAPI()`: Creates the API.
- `@app.get("/")`: Defines a GET endpoint.
- `main:app`: Python file and application instance.
- `--reload`: Automatically restarts the server when code changes.

**URLs:**
- API: `http://127.0.0.1:8000`
- API Docs: `http://127.0.0.1:8000/docs`
